In [19]:
from gen_catalyst_design.discrete_space_diffusion import ExponentialBetaScheduler, UniformTransitionsNoiser, AbsorbingStateNoiser, CosineScheduler
from gen_catalyst_design.stability import apply_inversion_symmetry
from gen_catalyst_design.utils import get_full_element_pool_no_saas
from gen_catalyst_design.discrete_space_diffusion.Dataset import get_dataloaders_from_atoms_list
from ase_ml_models.databases import get_atoms_list_from_db
from ase_ml_models.utilities import get_connectivity, plot_connectivity
from ase.db import connect
import torch
from ase.io import write
from ase.visualize import view
import os


In [25]:
miller_index = "100"
surface_type = "surface"
num_copies = 1
element_pool = get_full_element_pool_no_saas()
noiser_type = "Uniform"
timesteps = [0, 250, 500, 750, 1000]

scheduler = CosineScheduler(
        #beta_max=5e-2, 
        #beta_min=1e-4,
        time_sample_method="stratified"
    )

if noiser_type == "Absorbing":
    element_pool = ["(X)"] + element_pool

if noiser_type == "Absorbing":
    noiser = AbsorbingStateNoiser(
                element_pool=element_pool
    )
if noiser_type == "Uniform":
    noiser = UniformTransitionsNoiser(
        element_pool=element_pool
    )

noiser.pre_compute_accum_q_matrices(scheduler=scheduler)

ase_db = connect(f"../../databases/{surface_type}_templates/{miller_index}_templates.db")
template_atoms_list = get_atoms_list_from_db(ase_db)
template_atoms = template_atoms_list[0]
template_atoms.symbols = ["Au" for _ in range(len(template_atoms))]
atoms_list = [template_atoms.copy() for _ in range(num_copies)]

cell = template_atoms.get_cell()

train_loader, val_loader = get_dataloaders_from_atoms_list(
    atoms_list=atoms_list,
    element_pool=element_pool,
    batch_size=num_copies,
    train_val_split=0,
    do_initial_shuffling=False,
    do_train_shuffling=False,
)

tot_atoms_dict = {}
for timestep in timesteps:
    noised_samples = []
    for batch in train_loader:
        batch_copy = batch.clone()
        noiser.noise_batch_x0_xt(batch=batch_copy, time_batch=timestep*torch.ones(size=(batch_copy.num_nodes,), dtype=torch.long))
        for sample_idx in range(num_copies):
            graph = batch_copy.get_example(sample_idx)
            atoms = graph.to_atoms(element_pool)
            noised_samples.append(atoms)
    
    outdir = os.path.join(noiser_type, surface_type, miller_index)
    if not os.path.exists(outdir):
        os.makedirs(outdir)
    
    for i, view in enumerate([dict(), dict(rotation='10z,-75x')]):
        write(
            filename=os.path.join(outdir, f"{timestep}_view_{i}.png"),
            images=noised_samples,
            **view
        )

    if timestep == 750:
        symbols = noised_samples[0].get_chemical_symbols()
        adsorbates_dict = {
            "CO":20,
            "CO2":50
        }
        for adsorbate in adsorbates_dict:
            adsorbate_template = template_atoms_list[adsorbates_dict[adsorbate]].copy()
            adsorbate_template.set_cell(None)
            indices_ads = adsorbate_template.info["indices_ads"]
            for i, symbol in enumerate(symbols):
                if i not in indices_ads:
                    adsorbate_template[i].symbol = symbol
            write(
                filename=os.path.join(outdir, f"{adsorbate}.png"),
                images=adsorbate_template,
                **dict(rotation='10z,-75x')
            )
        if surface_type == "surface":
            atoms = template_atoms.copy()
            atoms.symbols = noised_samples[0].get_chemical_symbols()
            inv_atoms = apply_inversion_symmetry(
                atoms=atoms,
                miller_index=miller_index
            )
            inv_atoms.set_cell(None)
            write(
                filename=os.path.join(outdir, f"inverted.png"),
                images=inv_atoms,
                **dict(rotation='10z,-75x')
            )


In [ ]:
timesteps = [0, 250, 500, 750, 1000]
tot_atoms_dict = {}
for timestep in timesteps:
    noised_samples = []
    for batch in train_loader:
        batch_copy = batch.clone()
        noiser.noise_batch_x0_xt(batch=batch_copy, time_batch=timestep*torch.ones(size=(batch_copy.num_nodes,), dtype=torch.long))
        for sample_idx in range(num_copies):
            graph = batch_copy.get_example(sample_idx)
            atoms = graph.to_atoms(element_pool)
            noised_samples.append(atoms)
    tot_atoms_dict[timestep] = noised_samples

    

<Popen: returncode: None args: ['c:\\Users\\karst\\anaconda3\\envs\\cat_opt\...>